# pyGeoMapper — Notebook 4: 3D Geological Block Diagram, Cutaway & Fence Slices, and Interactive Viewer

**Site**: Sheep Mountain Anticline, Bighorn Basin, Wyoming, USA (`EPSG:32612`, UTM Zone 12N, `160 × 160` grid at `20 m` resolution)  
**Author**: Larry Syu-Heng Lai  
**License**: AGPL-3.0-or-later (`pyGeoMapper` is an independent Rust + Python geoscience toolkit and is distinct from `The-Magicians-Code/PyGeoMapper`.)

---

### Overview
This notebook demonstrates **`viewer_3d` + `pygeomapper._core` 3D mesh engine** of `pyGeoMapper`: building and visualizing **3D shallow geological block diagrams** with projected subsurface stratigraphy, planar/listric fault offsets, cutaway quadrants, intersecting 3D fence diagrams, binary glTF 2.0 (`.glb`) / `.ply` / `.obj` 3D mesh export, `forge3d` terrain draping, and a **100% offline interactive HTML5/WebGL 3D block viewer** (`outputs/sheep_mtn_3d_block_viewer.html`).

1. **Load Sheep Mountain Anticline Dataset & Auto-Mapped Units**:
   - USGS 3DEP DEM (`1134.0 m` to `1512.8 m` NAVD88, superposed Bighorn River canyon).
   - USGS SGMC Reference Formations (`0: Mm` Madison Limestone through `6: Qa` Quaternary Alluvium) and Supervised Hybrid Auto-Mapped Formations.
2. **Rust 3D Block Volume & Watertight Mesh Engine (`pygeomapper._core`)**:
   - Calibrates the 2D structural fold elevation surface $h_{\text{struct}}(y, x)$ from surface contacts, 3-point bed attitudes, and the steep SW forelimb reverse/thrust fault (`strike = 140°`, `dip = 58° NE`, `throw = 75 m`).
   - Evaluates the 3D voxel stratigraphic volume `(160 × 160 × 64)` down to `750 m` elevation (`~384 m` below the Bighorn River canyon floor) in parallel Rust (`build_3d_block_volume_rs`).
   - Generates complete 3D triangle meshes (`build_block_diagram_meshes_rs`) for the top topographic surface, 4 vertical cross-section side walls (South, North, West, East), base plate, internal cutaway walls/bench, and 3D thrust fault plane.
3. **3D Cutaway Block & Intersecting Fence Diagrams (`slice_3d_fence_diagram`)**:
   - Excavates the SW forelimb quadrant to expose the folded Paleozoic–Mesozoic core and thrust offset inside the 3D block.
   - Slices 3 intersecting vertical fence ribbons (`Fence A-A'` SW–NE dip panel, `Fence B-B'` NW–SE hinge panel, and `Fence C-C'` W–E panel).
4. **Multi-Modal Surface Drape Comparison & Interactive HTML5/WebGL 3D Viewer**:
   - Compares draping USGS Reference Geology, Auto-Mapped Geology, Sentinel-2 False-Color (`SWIR1-NIR-Red`), and 2D Mexican-Hat CWT Multi-Scale Complexity onto the 3D block diagram.
   - Exports binary `.glb`, `.ply`, and `.obj` 3D meshes and launches the self-contained interactive 3D HTML viewer (`outputs/sheep_mtn_3d_block_viewer.html`).

In [1]:
import json
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np

import pygeomapper as pgm
from pygeomapper import (
ROOT = Path.cwd() if (Path.cwd() / "example_data").exists() else Path.cwd().parent   # repo root (run from notebooks/ or the root)
    auto_mapping,
    cross_section,
    raster_io,
    terrain_analysis,
    viewer_3d,
)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 170

DATA_DIR = ROOT / "example_data"
OUT_DIR = ROOT / "outputs"
IMG_DIR = ROOT / "image"
for d in (OUT_DIR, IMG_DIR):
    d.mkdir(parents=True, exist_ok=True)

dem = raster_io.load_geotiff(DATA_DIR / "sheep_mtn_dem_3dep.tif", name="sheep_dem", units="m")
geol_ref = raster_io.load_geotiff(DATA_DIR / "sheep_mtn_geol_units.tif", name="sheep_geol", units="unit_id")
s2_stack = raster_io.load_multiband_geotiff(
    DATA_DIR / "sheep_mtn_s2.tif",
    band_names=["blue", "green", "red", "nir", "swir1", "swir2"],
    name="sheep_s2",
    units="reflectance",
)
with open(DATA_DIR / "sheep_mtn_units.json", encoding="utf-8") as f:
    units_meta = json.load(f)
unit_names = {u["unit_id"]: f"{u['code']}: {u['name']}" for u in units_meta["units"]}

print(f"pyGeoMapper version: {pgm.__version__} (forge3d available: {viewer_3d.HAS_FORGE3D})")
print(f"DEM grid: {dem.shape} at {dem.dx:.0f} m spacing, elevation {np.nanmin(dem.data):.1f} - {np.nanmax(dem.data):.1f} m NAVD88")
print(f"Formations loaded: {len(units_meta['units'])}")

pyGeoMapper version: 0.1.0 (forge3d available: True)
DEM grid: (160, 160) at 20 m spacing, elevation 1134.0 - 1512.8 m NAVD88
Formations loaded: 7


## 1. Reconstruct Toggleable Auto-Mapped Units (`feature_mode='engineered'` vs. `feature_mode='alphaearth'`)

### 1. What This Step Does & Why We Use This Method
- We load both the **29-band Engineered Physics + Spectral + 2D CWT stack** (`feature_mode='engineered'`) and the **64-band AlphaEarth Foundations Unit-Sphere Embedding stack** (`feature_mode='alphaearth'`, `sheep_mtn_alphaearth_2024.tif`) and train the hybrid supervised classifier on the spatial-block split.
- Both auto-mapped geological maps (`geol_auto_eng` and `geol_auto_ae`) are then built into 3D block models and added as **toggleable surface drapes** inside the interactive 3D HTML5/WebGL viewer (`outputs/sheep_mtn_3d_block_viewer.html`).


In [2]:
ae_stack = raster_io.load_multiband_geotiff(
    DATA_DIR / "sheep_mtn_alphaearth_2024.tif",
    dequantize_alphaearth=True,
    name="sheep_alphaearth",
)
feature_stack = auto_mapping.build_mapping_feature_stack(
    dem,
    spectral_stack=s2_stack,
    cwt_scales_m=[30.0, 60.0, 120.0, 200.0],
    feature_mode="engineered",
)
split = auto_mapping.create_spatial_block_split(
    dem,
    mode="west_east",
    test_fraction=0.35,
    buffer_px=2,
)
sup_eng = auto_mapping.train_predict_supervised_units(
    feature_stack,
    geol_ref,
    train_mask=split["train_mask"],
    method="hybrid",
    smooth_radius_px=2,
    feature_mode="engineered",
)
sup_ae = auto_mapping.train_predict_supervised_units(
    feature_stack,
    geol_ref,
    train_mask=split["train_mask"],
    method="hybrid",
    smooth_radius_px=2,
    embedding_stack=ae_stack,
    feature_mode="alphaearth",
)
geol_auto = sup_eng["pred_labels"]
geol_auto_ae = sup_ae["pred_labels"]

eval_eng_full = auto_mapping.evaluate_mapping_accuracy(geol_ref, geol_auto, unit_names=unit_names)
eval_ae_full = auto_mapping.evaluate_mapping_accuracy(geol_ref, geol_auto_ae, unit_names=unit_names)
eval_full = eval_ae_full
attitudes_3pt = cross_section.estimate_contact_attitudes(geol_ref, dem, window_m=320.0)
print(
    f"Engineered (29B) Full-Scene Agreement: {eval_eng_full['overall_accuracy']:.3f} | "
    f"AlphaEarth (64B) Full-Scene Agreement: {eval_ae_full['overall_accuracy']:.3f} | "
    f"3-Point Contact Attitudes: {len(attitudes_3pt)}"
)


Engineered (29B) Full-Scene Agreement: 0.883 | AlphaEarth (64B) Full-Scene Agreement: 0.955 | 3-Point Contact Attitudes: 41


## 2. Build the 3D Shallow Geological Block Model in Rust (`build_3d_block_diagram`)

### 1. What This Step Does & Why We Use This Method
- **The Goal:** Convert a 2D surface geological map, 3DEP Digital Elevation Model, and 3-point strike/dip attitudes into a watertight **3D volumetric block diagram** (`160 × 160 × 64` voxels down to `750 m` elevation) that reveals how folded beds and thrust faults project underground.

### 2. Governing Equations & Physical Meaning
- **3D Stratigraphic Potential Field (`calibrate_3d_structural_horizon` & `build_3d_block_volume_rs`):**
  $$\zeta(x, y, z) = z - h_{\text{struct}}(x, y) - \Delta z_{\text{fault}}(x, y, z)$$
  - $z$: Voxel elevation above sea level (meters NAVD88).
  - $h_{\text{struct}}(x, y)$: Smooth 2D structural fold reference surface calibrated from surface formation contacts and 3-point bedding attitudes.
  - $\Delta z_{\text{fault}}(x, y, z)$: Hanging-wall vertical throw ($+75\text{ m}$) above the dipping reverse/thrust fault plane $z_{\text{fault}}(x, y) = z_0 - d_{\text{dip}}\tan\delta_{\text{fault}}$.
  - Each voxel $(x, y, z)$ is assigned to formation $k$ according to its stratigraphic height $\zeta(x, y, z) \in [\zeta_{k-1}, \zeta_k)$.


In [3]:
xs = dem.x_coords()
ys = dem.y_coords()
fault_spec = {
    "name": "Sheep Mountain Forelimb Thrust",
    "x0": float(xs.min() + 0.34 * (xs.max() - xs.min())),
    "y0": float(ys.min() + 0.34 * (ys.max() - ys.min())),
    "z0": 1245.0,
    "strike_deg": 140.0,
    "dip_deg": 58.0,
    "throw_m": 75.0,
    "listric_curvature": 0.0,
}

block_ref = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_ref,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

block_auto_eng = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_auto,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

block_auto = viewer_3d.build_3d_block_diagram(
    dem=dem,
    geol_labels=geol_auto_ae,
    units_metadata=units_meta,
    attitudes=attitudes_3pt,
    z_bottom=750.0,
    nz=64,
    fault_spec=fault_spec,
    cutaway_fraction=(0.48, 0.48, 0.36),
)

print(f"Full 3D Block Mesh: {block_ref['mesh']['n_vertices']:,} vertices, {block_ref['mesh']['n_triangles']:,} triangles")
print(f"Cutaway 3D Block Mesh: {block_ref['cutaway_mesh']['n_vertices']:,} vertices, {block_ref['cutaway_mesh']['n_triangles']:,} triangles")
print("\nSubsurface 3D Voxel Volume Breakdown (Sheep Mountain Block down to 750 m):")
for st in block_ref["unit_volume_stats"]:
    print(
        f"  [{st['id']}] {st['code']:4s} ({st['name']:30s}): "
        f"{st['voxel_count']:7,d} voxels ({st['volume_fraction']*100:5.1f}%, {st['volume_km3']:.3f} km^3)"
    )


Full 3D Block Mesh: 128,271 vertices, 105,422 triangles
Cutaway 3D Block Mesh: 152,566 vertices, 107,702 triangles

Subsurface 3D Voxel Volume Breakdown (Sheep Mountain Block down to 750 m):
  [0] Mm   (Madison Limestone             ): 686,729 voxels ( 64.7%, 3.378 km^3)
  [1] IPta (Tensleep Sandstone & Amsden Fm): 147,772 voxels ( 13.9%, 0.727 km^3)
  [2] TRPc (Chugwater & Goose Egg Fms     ):  98,567 voxels (  9.3%, 0.485 km^3)
  [3] Jsg  (Sundance & Gypsum Spring Fms  ):  63,500 voxels (  6.0%, 0.312 km^3)
  [4] KJcm (Cloverly & Morrison Fms       ):  45,135 voxels (  4.2%, 0.222 km^3)
  [5] Kmt  (Mowry & Thermopolis Shales    ):  13,233 voxels (  1.2%, 0.065 km^3)
  [6] Qa   (Quaternary Alluvium & Terrace ):   7,092 voxels (  0.7%, 0.035 km^3)


## 3. Visualize 3D Geological Block Diagram: USGS Reference vs. Auto-Mapped Units (`image/3d_block_diagram.png`)

### How to Read the Visual Output
- **Top Surface:** Shows the 3D topography of Sheep Mountain Anticline and Bighorn Canyon draped with geological formation colors and black **3-point strike/dip pins**.
- **Four Vertical Side Walls:** Show the projected subsurface cross-sections along the South, West, North, and East boundaries down to $750\text{ m}$ elevation, revealing the arched **Madison Limestone (`Mm`, dark blue)** core beneath the younger flanking beds.


In [4]:
fig = plt.figure(figsize=(16, 7.2))
ax1 = fig.add_subplot(121, projection="3d")
viewer_3d.plot_3d_block_diagram(
    block_ref,
    vertical_exaggeration=1.7,
    azimuth_deg=-56.0,
    elevation_deg=28.0,
    top_stride=2,
    wall_stride=1,
    show_attitudes=True,
    show_legend=True,
    ax=ax1,
    title="(a) 3D Block Diagram — USGS Reference Geology + 3-Point Attitudes",
)

ax2 = fig.add_subplot(122, projection="3d")
viewer_3d.plot_3d_block_diagram(
    block_auto,
    vertical_exaggeration=1.7,
    azimuth_deg=-56.0,
    elevation_deg=28.0,
    top_stride=2,
    wall_stride=1,
    show_attitudes=True,
    show_legend=False,
    ax=ax2,
    title=f"(b) 3D Block Diagram — Auto-Mapped Geology (Full Agreement={eval_full['overall_accuracy']:.1%})",
)
plt.tight_layout()
fig_path1 = IMG_DIR / "3d_block_diagram.png"
fig.savefig(fig_path1, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path1}")

Saved image/3d_block_diagram.png


## 4. 3D Cutaway Block & Intersecting Structural Fence Diagram (`slice_3d_fence_diagram`)

To inspect the internal 3D geometry of the doubly plunging anticline and the SW forelimb thrust fault (`75 m` hanging-wall throw), we:
1. Render the **3D Cutaway Block** (`plot_3d_cutaway_block`) where the SW quadrant is excavated down to `~1025 m` elevation.
2. Slice **3 Intersecting 3D Fence Panels** (`Fence A-A'` SW–NE structural dip section, `Fence B-B'` NW–SE hinge section, and `Fence C-C'` W–E section) using `slice_3d_fence_diagram`.

In [5]:
fence_panels = viewer_3d.slice_3d_fence_diagram(block_ref, n_samples=160)
for fp in fence_panels:
    print(
        f"{fp['name']}: length={fp['distance_m'][-1]:.0f} m, "
        f"ribbon triangles={fp['mesh']['n_triangles']:,}"
    )

fig = plt.figure(figsize=(16, 7.2))
ax1 = fig.add_subplot(121, projection="3d")
viewer_3d.plot_3d_cutaway_block(
    block_ref,
    vertical_exaggeration=1.7,
    azimuth_deg=-52.0,
    elevation_deg=30.0,
    face_stride=1,
    show_fault_plane=True,
    show_legend=True,
    ax=ax1,
    title="(a) 3D Cutaway Block (SW Forelimb Excavated to Expose Fold & Thrust)",
)

ax2 = fig.add_subplot(122, projection="3d")
viewer_3d.plot_3d_fence_diagram(
    block_ref,
    fence_panels=fence_panels,
    vertical_exaggeration=1.7,
    azimuth_deg=-50.0,
    elevation_deg=28.0,
    show_fault_plane=True,
    show_wireframe_surface=True,
    show_legend=False,
    ax=ax2,
    title="(b) 3D Intersecting Structural Fence Diagram + Thrust Fault Surface",
)
plt.tight_layout()
fig_path2 = IMG_DIR / "3d_cutaway_and_fence.png"
fig.savefig(fig_path2, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path2}")

Fence A-A': length=3689 m, ribbon triangles=12,828
Fence B-B': length=3418 m, ribbon triangles=15,996
Fence C-C': length=2798 m, ribbon triangles=14,330


Saved image/3d_cutaway_and_fence.png


## 5. Multi-Modal Surface Drape Comparison & `forge3d` Terrain Adapter

A major advantage of separating the 3D subsurface block mesh from the surface texture is that any co-registered remote-sensing or geomorphic layer can be draped over the 3D topography while preserving the projected subsurface geology on the 4 vertical side walls:
1. **USGS Reference Geological Map** (`block_ref['surface_drape_rgba']`)
2. **Supervised Auto-Mapped Formations** (`block_auto['surface_drape_rgba']`)
3. **Sentinel-2 False-Color Lithologic Composite (`SWIR1 - NIR - Red`)**
4. **2D Mexican-Hat CWT Multi-Scale Topographic Complexity (`pyTopoComplexity`)**

In [6]:
def stretch_band(arr: np.ndarray) -> np.ndarray:
    lo, hi = np.nanpercentile(arr, [2.0, 98.0])
    return np.clip((arr - lo) / max(hi - lo, 1e-9), 0.0, 1.0)

hs = block_ref["hillshade"]
s2_false_rgb = np.stack(
    [
        stretch_band(s2_stack.band("swir1").data),
        stretch_band(s2_stack.band("nir").data),
        stretch_band(s2_stack.band("red").data),
    ],
    axis=-1,
)
# Blend Sentinel-2 false color with multi-azimuth hillshade
s2_drape_f = np.clip(s2_false_rgb * (0.48 + 0.72 * hs.data[:, :, np.newaxis]), 0.0, 1.0)
s2_drape_rgba = np.dstack(
    [(s2_drape_f * 255).astype(np.uint8), np.full(dem.shape, 255, dtype=np.uint8)]
)

# Drape 2D CWT Multi-Scale Complexity on hillshade via forge3d adapter
cwt_mean_grid = feature_stack.band("mean_cwt_complexity")
cwt_drape_f = viewer_3d.drape_overlay_on_hillshade(
    hs, cwt_mean_grid, cmap="magma", alpha=0.75
)
cwt_drape_rgba = np.dstack(
    [(cwt_drape_f * 255).astype(np.uint8), np.full(dem.shape, 255, dtype=np.uint8)]
)
forge_info = viewer_3d.render_with_forge3d(dem, overlay_rgb=cwt_drape_f)
print(f"3D Terrain Drape Backend: {forge_info['backend']} (HAS_FORGE3D={forge_info['has_forge3d']})")

fig = plt.figure(figsize=(15.5, 12.0))
drape_configs = [
    ("(a) USGS SGMC Reference Geology Drape", block_ref["surface_drape_rgba"]),
    ("(b) Auto-Mapped Formations Drape", block_auto["surface_drape_rgba"]),
    ("(c) Sentinel-2 Lithologic False-Color (SWIR1-NIR-Red) Drape", s2_drape_rgba),
    ("(d) 2D Mexican-Hat CWT Multi-Scale Complexity Drape", cwt_drape_rgba),
]
for idx, (sub_title, d_rgba) in enumerate(drape_configs, start=1):
    ax = fig.add_subplot(2, 2, idx, projection="3d")
    viewer_3d.plot_3d_block_diagram(
        block_ref,
        surface_drape_rgba=d_rgba,
        vertical_exaggeration=1.65,
        azimuth_deg=-55.0,
        elevation_deg=29.0,
        top_stride=2,
        wall_stride=1,
        show_attitudes=(idx <= 2),
        show_legend=(idx == 1),
        ax=ax,
        title=sub_title,
    )
plt.tight_layout()
fig_path3 = IMG_DIR / "surface_drape_comparison.png"
fig.savefig(fig_path3, bbox_inches="tight")
plt.show()
print(f"Saved {fig_path3}")

3D Terrain Drape Backend: forge3d (HAS_FORGE3D=True)


Saved image/surface_drape_comparison.png


## 6. Export 3D Meshes (`.glb`, `.ply`, `.obj`) & Launch Offline Interactive 3D Block Viewer (`outputs/sheep_mtn_3d_block_viewer.html`)

### Toggleable Surface Drapes in the Interactive 3D Viewer
Both mapping methods remain **first-class toggleable options** inside the interactive HTML5/WebGL viewer dropdown:
1. `Geological Map (Primary)` — USGS SGMC Reference Map
2. `Auto-Mapped Units (Engineered 29-Band)` — Physics + Spectral + 2D CWT Auto-Map (`feature_mode='engineered'`)
3. `Auto-Mapped Units (AlphaEarth 64-Band)` — AlphaEarth Unit-Sphere Embedding Auto-Map (`feature_mode='alphaearth'`)
4. `Sentinel-2 False-Color (SWIR1-NIR-Red)` — Optical lithologic composite
5. `2D CWT Multi-Scale Complexity` — Multi-scale wavelet energy drape


In [7]:
glb_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_block.glb", use_cutaway=False, vertical_exaggeration=1.5
)
ply_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_cutaway.ply", use_cutaway=True, vertical_exaggeration=1.5
)
obj_res = viewer_3d.export_3d_block_mesh(
    block_ref, OUT_DIR / "sheep_mtn_block.obj", use_cutaway=False, vertical_exaggeration=1.5
)
print("Exported 3D Meshes:")
for r in (glb_res, ply_res, obj_res):
    print(
        f"  - {r['path']} ({r['format'].upper()}): {r['n_vertices']:,} vertices, "
        f"{r['n_triangles']:,} triangles, {r['file_size_bytes']/1024:.1f} KB"
    )

extra_drapes = {
    "Auto-Mapped Units (Engineered 29-Band)": block_auto_eng["surface_drape_rgba"],
    "Auto-Mapped Units (AlphaEarth 64-Band)": block_auto["surface_drape_rgba"],
    "Sentinel-2 False-Color (SWIR1-NIR-Red)": s2_drape_rgba,
    "2D CWT Multi-Scale Complexity": cwt_drape_rgba,
}
html_meta = viewer_3d.export_interactive_block_html(
    block_ref,
    output_html=OUT_DIR / "sheep_mtn_3d_block_viewer.html",
    extra_surface_drapes=extra_drapes,
    grid_stride=2,
    title="Sheep Mountain Anticline, WY — 3D Geological Block & Subsurface Viewer",
)
print(
    f"\nInteractive 3D HTML Viewer generated: {html_meta['html_path']} "
    f"({html_meta['file_size_bytes']/1024:.1f} KB, drapes={html_meta['surface_drapes']})"
)
viewer_3d.launch_block_viewer(
    block_ref,
    output_html=OUT_DIR / "sheep_mtn_3d_block_viewer.html",
    extra_surface_drapes=extra_drapes,
    grid_stride=2,
)


Exported 3D Meshes:
  - outputs/sheep_mtn_block.glb (GLB): 128,271 vertices, 105,422 triangles, 4743.8 KB
  - outputs/sheep_mtn_cutaway.ply (PLY): 152,566 vertices, 107,702 triangles, 11417.8 KB
  - outputs/sheep_mtn_block.obj (OBJ): 128,271 vertices, 105,422 triangles, 13423.2 KB

Interactive 3D HTML Viewer generated: outputs/sheep_mtn_3d_block_viewer.html (1015.4 KB, drapes=['Geological Map (Primary)', 'Auto-Mapped Units (Engineered 29-Band)', 'Auto-Mapped Units (AlphaEarth 64-Band)', 'Sentinel-2 False-Color (SWIR1-NIR-Red)', '2D CWT Multi-Scale Complexity'])
